# 이전 대화를 기억하는 multi-turn agent

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

LangChain v1의 고수준 권장 API인 `create_agent`에 checkpointer를 전달하면 대화형
agent의 단기 메모리를 구성할 수 있습니다. 내부 실행은 LangGraph 기반이며,
호출할 때 같은 `thread_id`를 사용하면 이전 상태가 자동으로 복원됩니다.

이 노트북은 과거의 `ConversationChain`과 별도 `langchain_teddynote` 로깅 helper 없이
multi-turn 대화, 세션 격리, 상태 조회를 구현합니다.


In [1]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" python-dotenv


## 환경 설정

`.env`에 `OPENAI_API_KEY`를 설정합니다. LangSmith 추적이 필요하면 코드 변경 없이
다음 환경 변수를 추가할 수 있습니다.

```text
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=CH05-Memory
```


In [2]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.6-luna")

agent = create_agent(
    model=MODEL_ID,
    tools=[],
    system_prompt=(
        "당신은 Question-Answering 챗봇입니다. "
        "현재 thread의 대화 기록을 활용해 정확하고 간결하게 답하세요."
    ),
    checkpointer=InMemorySaver(),
)


## 같은 thread에서 multi-turn 대화


In [3]:
def ask(question: str, thread_id: str) -> str:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    return result["messages"][-1].content


In [4]:
print(ask("나의 이름은 테디입니다.", thread_id="abc123"))


알겠습니다, 테디님.


In [5]:
print(ask("내 이름이 뭐라고?", thread_id="abc123"))


테디님입니다.


## 다른 thread는 별도 대화


In [6]:
print(ask("내 이름이 뭐라고?", thread_id="abc1234"))


아직 이름을 알려주지 않으셔서 알 수 없습니다.


## 현재 thread 상태 조회

checkpointer의 상태는 `get_state()`로 읽습니다. 레거시 메모리 객체의 내부 필드에
직접 접근할 필요가 없습니다.


In [7]:
config = {"configurable": {"thread_id": "abc123"}}
snapshot = agent.get_state(config)

for message in snapshot.values["messages"]:
    print(f"[{message.type}] {message.content}")


[human] 나의 이름은 테디입니다.
[ai] 알겠습니다, 테디님.
[human] 내 이름이 뭐라고?
[ai] 테디님입니다.


여기서 만든 메모리는 **thread 내부 단기 메모리**입니다. 여러 thread를 넘어 사용자
선호나 프로필을 기억하려면 checkpointer가 아니라 LangGraph Store 기반 장기 메모리를
함께 사용하세요(04, 07번 노트북 참고).
